In [ ]:
%load_ext autoreload
%autoreload 2

import time

import matplotlib.pyplot as plt
import numpy as np

from rogii_wellbore.data import list_wells, load_horizontal, load_typewell
from rogii_wellbore.features import _gr_interpolate, pick_inference_anchor

TEST_WELLS = {"000d7d20", "00bbac68", "00e12e8b"}

# Pick first train well that's not in the test set.
all_train = list_wells("train")
WELL_ID = next(w for w in all_train if w not in TEST_WELLS)
print(f"Exploring well: {WELL_ID}")

In [ ]:
well = load_horizontal("train", well_ids=[WELL_ID], source="parquet").reset_index(drop=True)
tw = load_typewell("train", well_ids=[WELL_ID], source="parquet").reset_index(drop=True)

print("well cols:", list(well.columns))
print("tw cols:  ", list(tw.columns))
print(f"\nwell rows: {len(well)}, tw rows: {len(tw)}")
print("\ntypewell head:")
print(tw.head())

tw_tvt_raw = tw["TVT"].to_numpy(dtype=float)
tw_gr_raw = tw["GR"].to_numpy(dtype=float)
print(f"\ntw TVT range: {tw_tvt_raw.min():.1f} to {tw_tvt_raw.max():.1f}")
print(f"tw native step (first 5 diffs): {np.diff(tw_tvt_raw)[:5]}")

# Resample typewell to 1.0 ft TVT grid (lateral MD step is 1.0 ft; matching grids = clean cross-correlation)
order = np.argsort(tw_tvt_raw)
tw_tvt_sorted, tw_gr_sorted = tw_tvt_raw[order], tw_gr_raw[order]

# Handle any NaN in typewell GR before interpolating onto the grid
finite = np.isfinite(tw_gr_sorted)
if not finite.all():
    print(f"WARNING: typewell has {(~finite).sum()} NaN GR rows; dropping before resample")
    tw_tvt_sorted, tw_gr_sorted = tw_tvt_sorted[finite], tw_gr_sorted[finite]

tvt_grid = np.arange(np.ceil(tw_tvt_sorted.min()), np.floor(tw_tvt_sorted.max()) + 1.0, 1.0)
gr_grid = np.interp(tvt_grid, tw_tvt_sorted, tw_gr_sorted)
print(f"\nresampled typewell: {len(tvt_grid)} rows, TVT {tvt_grid[0]:.1f} to {tvt_grid[-1]:.1f}")

In [ ]:
def gr_match_mvp(
    lateral_gr: np.ndarray,
    typewell_tvt: np.ndarray,
    typewell_gr: np.ndarray,
    anchor_tvt: float,
    window_len: int = 101,
    radius: float = 75.0,
    window_shape: str = "centered",  # "centered" or "causal"
) -> tuple[np.ndarray, np.ndarray]:
    """Slow, correct MVP. Returns (gr_match_tvt, gr_match_sim), len = len(lateral_gr)."""
    n = len(lateral_gr)
    out_tvt = np.full(n, np.nan)
    out_sim = np.full(n, np.nan)

    if window_shape == "centered":
        left = window_len // 2
        right = window_len - left
    elif window_shape == "causal":
        left, right = window_len - 1, 1
    else:
        raise ValueError(window_shape)

    cand_j = np.flatnonzero(np.abs(typewell_tvt - anchor_tvt) <= radius)
    if len(cand_j) == 0:
        return out_tvt, out_sim

    for i in range(n):
        lo, hi = i - left, i + right
        if lo < 0 or hi > n:
            continue
        lat = lateral_gr[lo:hi]
        if np.isnan(lat).any():
            continue
        a = lat - lat.mean()
        a_norm = np.sqrt((a * a).sum())
        if a_norm == 0:
            continue

        best_sim, best_tvt = -np.inf, np.nan
        for j in cand_j:
            tj_lo, tj_hi = j - left, j + right
            if tj_lo < 0 or tj_hi > len(typewell_gr):
                continue
            tw_w = typewell_gr[tj_lo:tj_hi]
            b = tw_w - tw_w.mean()
            b_norm = np.sqrt((b * b).sum())
            if b_norm == 0:
                continue
            sim = (a * b).sum() / (a_norm * b_norm)
            if sim > best_sim:
                best_sim, best_tvt = sim, typewell_tvt[j]

        if np.isfinite(best_sim):
            out_tvt[i] = best_tvt
            out_sim[i] = best_sim

    return out_tvt, out_sim

In [ ]:
lat_gr = _gr_interpolate(well["GR"].to_numpy(dtype=float))
anchor_idx = pick_inference_anchor(well)
anchor_tvt = well["TVT_input"].to_numpy(dtype=float)[anchor_idx]
print(
    f"anchor_idx={anchor_idx}, anchor_tvt={anchor_tvt:.2f}, "
    f"n_rows={len(well)}, eval_rows={len(well) - anchor_idx - 1}"
)

t0 = time.time()
tvt_c, sim_c = gr_match_mvp(lat_gr, tvt_grid, gr_grid, anchor_tvt, window_shape="centered")
print(f"centered: {time.time() - t0:.1f}s")

t0 = time.time()
tvt_z, sim_z = gr_match_mvp(lat_gr, tvt_grid, gr_grid, anchor_tvt, window_shape="causal")
print(f"causal:   {time.time() - t0:.1f}s")

In [ ]:
true_tvt = well["TVT"].to_numpy(dtype=float)
row = np.arange(len(well))
eval_mask = np.isnan(well["TVT_input"].to_numpy(dtype=float))

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]})

ax = axes[0]
ax.plot(row, true_tvt, color="black", lw=1.2, label="true TVT")
ax.plot(row[eval_mask], tvt_c[eval_mask], ".", ms=2.5, alpha=0.7, label="gr_match (centered)")
ax.plot(row[eval_mask], tvt_z[eval_mask], ".", ms=2.5, alpha=0.7, label="gr_match (causal)")
ax.axvline(anchor_idx, color="red", ls="--", alpha=0.5, label="anchor")
ax.axhline(anchor_tvt, color="red", ls=":", alpha=0.3)
ax.set_ylabel("TVT")
ax.set_title(f"{WELL_ID}: gr_match prediction vs true TVT (eval zone only)")
ax.legend(loc="best")

ax = axes[1]
ax.plot(row[eval_mask], sim_c[eval_mask], ".", ms=2.5, alpha=0.7, label="centered")
ax.plot(row[eval_mask], sim_z[eval_mask], ".", ms=2.5, alpha=0.7, label="causal")
ax.set_ylabel("gr_match_sim")
ax.set_xlabel("row_idx")
ax.set_ylim(-0.2, 1.05)
ax.legend(loc="best")

plt.tight_layout()
plt.savefig("../reports/figures/06_gr_match_eval.png", dpi=100, bbox_inches="tight")
plt.show()

# Quick numeric summary on eval rows
for name, pred in [("centered", tvt_c), ("causal", tvt_z)]:
    m = eval_mask & np.isfinite(pred)
    rmse = np.sqrt(((pred[m] - true_tvt[m]) ** 2).mean()) if m.any() else float("nan")
    cov = m.sum() / eval_mask.sum()
    print(f"{name:8s} eval RMSE = {rmse:.3f}   coverage = {cov:.1%}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6))

# Top: lateral GR through full well
ax = axes[0]
ax.plot(row, lat_gr, lw=0.6, color="steelblue")
ax.axvline(anchor_idx, color="red", ls="--", alpha=0.5, label="anchor")
ax.set_ylabel("lateral GR")
ax.set_title(f"{WELL_ID}: lateral GR (full well)")
ax.legend()

# Bottom: typewell GR through the search range, with true-eval-TVT range overlaid
ax = axes[1]
mask_search = np.abs(tvt_grid - anchor_tvt) <= 75
ax.plot(tvt_grid[mask_search], gr_grid[mask_search], lw=0.8, color="darkorange")
ax.axvline(anchor_tvt, color="red", ls="--", alpha=0.5, label="anchor TVT")
true_eval_tvt = true_tvt[eval_mask]
ax.axvspan(
    np.nanmin(true_eval_tvt),
    np.nanmax(true_eval_tvt),
    color="green",
    alpha=0.15,
    label=f"true eval TVT range [{np.nanmin(true_eval_tvt):.0f}, {np.nanmax(true_eval_tvt):.0f}]",
)
ax.set_xlabel("typewell TVT")
ax.set_ylabel("typewell GR")
ax.set_title(f"typewell GR across search range (\u00b175 ft of anchor {anchor_tvt:.1f})")
ax.legend()

plt.tight_layout()
plt.savefig("../reports/figures/07_lateral_gr.png", dpi=100, bbox_inches="tight")
plt.show()

# Also: lateral GR scale vs typewell GR scale — are they even comparable?
print(
    f"lateral GR: mean={np.nanmean(lat_gr):.1f}, std={np.nanstd(lat_gr):.1f}, "
    f"range=[{np.nanmin(lat_gr):.1f}, {np.nanmax(lat_gr):.1f}]"
)
print(
    f"typewell GR (search range): mean={gr_grid[mask_search].mean():.1f}, std={gr_grid[mask_search].std():.1f}, "
    f"range=[{gr_grid[mask_search].min():.1f}, {gr_grid[mask_search].max():.1f}]"
)